# Daily weather from the German Weather Service (DWD)

**Purpose.** Download the official daily weather observations of the DWD station **Kiel-Holtenau (station ID 02564)** and save them as `dwd_weather.csv` in this folder, so that all notebooks can join them on `Datum`.

**Why?** The course file `wetter.csv` has no measured rain or sunshine, and its `Wettercode` is missing on 669 days. A comparison (2026-11-24) showed that the missing codes do **not** mean "clear sky": on those days the DWD measured rain on 23 % and an average cloud cover of 3.6 eighths. DWD measures rain amount and sunshine hours directly and has almost no gaps.

**Source.** DWD Climate Data Center, daily climate data (`KL`), historical (quality-controlled) files: <https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/>. Licence: free use with source attribution (DWD open data, CC BY 4.0).

**Period saved.** 2012-01-01 – 2019-08-01, the same span as the course weather file (training **and** test period).


In [1]:
import io
import re
import zipfile
from pathlib import Path

import pandas as pd
import requests

DWD_DIR = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/"
STATION = "02564"                       # Kiel-Holtenau
START, END = "2012-01-01", "2019-08-01"  # same span as the course file wetter.csv
OUT_FILE = Path("dwd_weather.csv")
COURSE_URL = "https://raw.githubusercontent.com/opencampus-sh/intro-to-data-science-and-ml/main/data/bakery/"



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\tping\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\tping\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\tping\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\tping\anaconda3\Lib\site-packa

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\tping\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\tping\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\tping\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\tping\anaconda3\Lib\site-packa

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\tping\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\tping\anaconda3\Lib\site-packages\traitlets\config\application.py", line 992, in launch_instance
    app.start()
  File "C:\Users\tping\anaconda3\Lib\site-packages\ipykernel\kernelapp.py", line 701, in start
    self.io_loop.start()
  File "C:\Users\tping\anaconda3\Lib\site-packa

AttributeError: _ARRAY_API not found

## Download

The name of the zip file contains the last date of the record and changes when DWD updates it, so the notebook looks it up in the directory listing first. The zip is read in memory; nothing is unpacked to disk.

In [2]:
listing = requests.get(DWD_DIR, timeout=60).text
zip_name = sorted(set(re.findall(rf"tageswerte_KL_{STATION}_\d+_\d+_hist\.zip", listing)))[-1]
print("file:", zip_name)

archive = zipfile.ZipFile(io.BytesIO(requests.get(DWD_DIR + zip_name, timeout=120).content))
data_file = next(n for n in archive.namelist() if n.startswith("produkt_klima_tag"))
raw = pd.read_csv(archive.open(data_file), sep=";", skipinitialspace=True, na_values=[-999])
raw.columns = raw.columns.str.strip()
print("rows in the full record:", len(raw), "| columns:", raw.columns.tolist())


file: tageswerte_KL_02564_19740101_20251231_hist.zip
rows in the full record: 18854 | columns: ['STATIONS_ID', 'MESS_DATUM', 'QN_3', 'FX', 'FM', 'QN_4', 'RSK', 'RSKF', 'SDK', 'SHK_TAG', 'NM', 'VPM', 'PM', 'TMK', 'UPM', 'TXK', 'TNK', 'TGK', 'eor']


## Select, rename and cut to the project period

DWD uses short codes for the columns. We keep the measured values, give them readable names and drop the quality-flag columns (`QN_3`, `QN_4`) and the end-of-record marker. `-999` means "not measured" and becomes an empty value.

| New name | DWD | Meaning | Unit | Measured over |
|---|---|---|---|---|
| `Niederschlag_mm` | RSK | precipitation amount | mm | **06:00 UTC on this day – 06:00 UTC next day** |
| `Niederschlagsform` | RSKF | form of precipitation (code, see below) | – | same as RSK |
| `Sonnenschein_h` | SDK | sunshine duration | hours | 00–24 UTC |
| `Bewoelkung_dwd` | NM | mean cloud cover | eighths (0–8) | daily mean |
| `Temperatur_dwd` | TMK | mean air temperature (2 m) | °C | daily mean |
| `Temperatur_max` | TXK | maximum air temperature | °C | 00–24 UTC |
| `Temperatur_min` | TNK | minimum air temperature | °C | 00–24 UTC |
| `Luftfeuchte` | UPM | mean relative humidity | % | daily mean |
| `Wind_mittel` | FM | mean wind speed | m/s | daily mean |
| `Wind_boe_max` | FX | strongest gust | m/s | daily |
| `Luftdruck` | PM | mean air pressure | hPa | daily mean |
| `Schneehoehe_cm` | SHK_TAG | snow depth | cm | at 06 UTC |

`Niederschlagsform` codes (DWD): 0 = no precipitation, 1 = only rain (before 1979), 4 = precipitation of unknown form, 6 = only rain, 7 = only snow, 8 = rain and snow, 9 = not known. **Use `Niederschlag_mm` for "did it rain", not the form code.**

**Watch out:** the precipitation of a day runs until 06:00 UTC of the *next* morning, so it includes the following night. The shop hours of the day are covered; rain in the night after closing is counted too.

The names `Bewoelkung_dwd` and `Temperatur_dwd` carry the suffix `_dwd` so they do not clash with `Bewoelkung` / `Temperatur` in the course file.

In [3]:
RENAME = {
    "RSK": "Niederschlag_mm", "RSKF": "Niederschlagsform", "SDK": "Sonnenschein_h", "NM": "Bewoelkung_dwd",
    "TMK": "Temperatur_dwd", "TXK": "Temperatur_max", "TNK": "Temperatur_min", "UPM": "Luftfeuchte",
    "FM": "Wind_mittel", "FX": "Wind_boe_max", "PM": "Luftdruck", "SHK_TAG": "Schneehoehe_cm",
}
dwd = raw.assign(Datum=pd.to_datetime(raw["MESS_DATUM"].astype(str), format="%Y%m%d"))
dwd = dwd[["Datum", *RENAME]].rename(columns=RENAME)
dwd = dwd[dwd["Datum"].between(START, END)].reset_index(drop=True)
dwd["Niederschlagsform"] = dwd["Niederschlagsform"].astype("Int64")

def gap_ranges(dates):
    """Group a sorted list of dates into runs of consecutive days: (first, last, number of days)."""
    dates = pd.Series(sorted(dates))
    run = (dates.diff() != pd.Timedelta(days=1)).cumsum()
    return [(g.min().date(), g.max().date(), len(g)) for _, g in dates.groupby(run)]

all_days = pd.date_range(START, END)
print("days:", len(dwd), "of", len(all_days), "| dates unique:", dwd["Datum"].is_unique)
print("calendar days without any DWD row (first, last, days):")
for gap in gap_ranges(all_days.difference(dwd["Datum"])):
    print("  ", gap)

days: 2631 of 2770 | dates unique: True
calendar days without any DWD row (first, last, days):
   (datetime.date(2012, 11, 3), datetime.date(2012, 11, 3), 1)
   (datetime.date(2012, 11, 10), datetime.date(2012, 11, 10), 1)
   (datetime.date(2012, 11, 17), datetime.date(2012, 11, 17), 1)
   (datetime.date(2012, 11, 24), datetime.date(2012, 11, 24), 1)
   (datetime.date(2012, 12, 1), datetime.date(2012, 12, 2), 2)
   (datetime.date(2012, 12, 8), datetime.date(2012, 12, 8), 1)
   (datetime.date(2012, 12, 15), datetime.date(2012, 12, 15), 1)
   (datetime.date(2012, 12, 21), datetime.date(2013, 4, 30), 131)


**What to check:** which days and columns are empty, and above all whether the gaps fall into the **sales (training)** or the **test** period. Gaps before July 2013 do not matter for the model, because there are no sales data then.

In [4]:
PERIODS = {"before sales (2012-01 – 2013-06)": ("2012-01-01", "2013-06-30"),
           "sales / training (2013-07 – 2018-07)": ("2013-07-01", "2018-07-31"),
           "test (2018-08 – 2019-07)": ("2018-08-01", "2019-07-31")}
calendar = pd.DataFrame({"Datum": all_days}).merge(dwd, on="Datum", how="left")   # every calendar day, gaps as empty
coverage = pd.DataFrame({name: calendar[calendar["Datum"].between(a, b)].drop(columns="Datum").isna().sum()
                         for name, (a, b) in PERIODS.items()})
coverage.loc["(days in period)"] = [calendar["Datum"].between(a, b).sum() for a, b in PERIODS.values()]
print("empty values per column and period (missing calendar days count as empty):")
coverage

empty values per column and period (missing calendar days count as empty):


,before sales (2012-01 – 2013-06),sales / training (2013-07 – 2018-07),test (2018-08 – 2019-07)
Niederschlag_mm,140,0,0
Niederschlagsform,140,0,0
Sonnenschein_h,212,0,88
Bewoelkung_dwd,171,11,25
Temperatur_dwd,145,0,0
Temperatur_max,180,0,0
Temperatur_min,180,0,0
Luftfeuchte,183,11,27
Wind_mittel,488,1,22
Wind_boe_max,488,0,21


In [5]:
for col in ["Sonnenschein_h", "Bewoelkung_dwd"]:
    empty = calendar.loc[calendar[col].isna() & calendar["Datum"].ge("2013-07-01"), "Datum"]
    print(f"{col}: empty from July 2013 on (first, last, days):", gap_ranges(empty))

Sonnenschein_h: empty from July 2013 on (first, last, days): [(datetime.date(2019, 5, 5), datetime.date(2019, 8, 1), 89)]
Bewoelkung_dwd: empty from July 2013 on (first, last, days): [(datetime.date(2017, 10, 30), datetime.date(2017, 11, 9), 11), (datetime.date(2018, 8, 6), datetime.date(2018, 8, 6), 1), (datetime.date(2018, 9, 8), datetime.date(2018, 9, 9), 2), (datetime.date(2018, 9, 13), datetime.date(2018, 9, 13), 1), (datetime.date(2018, 9, 15), datetime.date(2018, 9, 21), 7), (datetime.date(2018, 10, 11), datetime.date(2018, 10, 15), 5), (datetime.date(2018, 10, 18), datetime.date(2018, 10, 18), 1), (datetime.date(2018, 10, 20), datetime.date(2018, 10, 24), 5), (datetime.date(2019, 4, 4), datetime.date(2019, 4, 4), 1), (datetime.date(2019, 6, 13), datetime.date(2019, 6, 14), 2)]


**Result (run 2026-11-24).**
- **Training period (Jul 2013 – Jul 2018): rain, sunshine and temperature are complete.** Cloud cover and humidity are empty on 11 days (2017-10-30 – 2017-11-09).
- **Test period (Aug 2018 – Jul 2019): rain and temperature are complete.** **Sunshine is missing from 2019-05-05 on (88 test days):** DWD stopped measuring sunshine at Kiel-Holtenau then (the DWD metadata list sunshine only until 2019-05-04). Cloud cover is empty on 25 test days.
- Snow depth is empty on many days (850 in training) – DWD seems to leave it blank instead of writing 0 when there is no snow. Do not use it without checking.
- Before the sales period, a whole block is missing (2012-12-21 – 2013-04-30, plus single Saturdays in Nov/Dec 2012). This does not affect the model.

**Open for the team:** sunshine can only be used as a feature if the 88 test days get a value from somewhere else (e.g. a nearby station), or if it is dropped. Rain (`Niederschlag_mm`) has no such problem.

## Plausibility: does DWD describe the same place as the course file?

If both sources measure Kiel, daily temperature and cloud cover should move together closely.

In [6]:
course = pd.read_csv(COURSE_URL + "wetter.csv", parse_dates=["Datum"])
both = course.merge(dwd, on="Datum", how="inner")
print("common days:", len(both))
print("correlation temperature:", round(both["Temperatur"].corr(both["Temperatur_dwd"]), 3))
print("correlation cloud cover:", round(both["Bewoelkung"].corr(both["Bewoelkung_dwd"]), 3))
print("days in DWD but not in the course file:", len(set(dwd["Datum"]) - set(course["Datum"])))


common days: 2599
correlation temperature: 0.987
correlation cloud cover: 0.887
days in DWD but not in the course file: 32


## Save

Saved with `Datum` as `YYYY-MM-DD`, the same name and format as in the course files, so the file joins directly (left join on `Datum`). The DWD record is not changed here: no filling of gaps and no features. Turning it into model features (for example a rain yes/no flag) happens later, in the feature step; the threshold for "rain" is a team decision.

In [7]:
dwd.to_csv(OUT_FILE, index=False, date_format="%Y-%m-%d")
print("saved", OUT_FILE, "with", len(dwd), "rows and", dwd.shape[1], "columns")
dwd.describe().round(1).T


saved dwd_weather.csv with 2631 rows and 13 columns


,count,mean,min,25%,50%,75%,max,std
Datum,2631,2015-12-06 17:35:14.025085,2012-01-01 00:00:00,2014-03-07 12:00:00,2015-12-25 00:00:00,2017-10-12 12:00:00,2019-08-01 00:00:00,NaN
Niederschlag_mm,2630.0,2.1,0.0,0.0,0.1,2.2,61.6,4.2
Niederschlagsform,2630.0,3.2,0.0,0.0,4.0,4.0,8.0,2.3
Sonnenschein_h,2469.0,4.7,0.0,0.3,3.6,7.9,16.6,4.6
Bewoelkung_dwd,2563.0,5.0,0.0,3.5,5.5,7.0,8.0,2.4
Temperatur_dwd,2625.0,10.2,-11.9,5.3,10.1,15.5,26.9,6.3
Temperatur_max,2590.0,13.9,-8.9,8.2,13.9,19.9,35.1,7.4
Temperatur_min,2590.0,6.6,-18.1,2.4,6.7,11.1,21.2,5.7
Luftfeuchte,2549.0,81.3,48.0,75.0,82.0,89.0,100.0,9.6
Wind_mittel,2259.0,3.8,1.0,2.6,3.5,4.6,12.0,1.5
